# How the PPMI tables relate

This notebook maps every table you have loaded: the keys that join them, how well those keys match, which participants and visits they share, and how the clean layer is built from the raw tables. It reads the downloads described in `dataset.ipynb`, works with whatever subset you have, and prints counts only.

PPMI's tables hang off three keys:

- `PATNO`, the participant. `Participant_Status` is the registry, with one row per participant, their cohort and their enrolment status.
- `PATNO` with `EVENT_ID`, the visit: SC (screening), BL (baseline), V01, V02 and onwards, and codes such as LOG for running logs. Assessments made at the same visit join on both.
- `IMAGEID`, the image. Derived imaging tables carry the ID of the scan they were measured on, and image searches and downloads list the same IDs.

Dates are recorded to the month, so a visit code and a month are the two ways to line measurements up in time.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import sys
sys.path.append("../code")

import data
import plot
from ppmi_pipeline import pipeline, sources

p = data.PPMI()
tables = sorted(p.files) + ["images", "image_files"]
columns = {name: set(p.db.table(name).columns) for name in tables}

## 1. Keys and grain

For each raw table: its key columns, its rows, participants and participant-visits, and how many rows it has per participant-visit. A visit table has about one row per participant-visit; a log or a scan table can have more.

In [ ]:
KEYS = {"PATNO": "PATNO", "EVENT_ID": "EVENT_ID", "event": "EVENT_ID", "IMAGEID": "IMAGEID", "image_id": "IMAGEID", "REC_ID": "REC_ID", "INFODT": "INFODT"}


def grain(position, name):
    """One row of the profile: the keys a table has, and its rows, participants and participant-visits."""
    keys = {KEYS[column] for column in columns[name] if column in KEYS}
    visit = next((column for column in ("EVENT_ID", "event") if column in columns[name]), None)
    flags = ", ".join(("'yes'" if key in keys else "''") + f' AS "{key}"' for key in dict.fromkeys(KEYS.values()))
    participants = "count(DISTINCT PATNO)" if "PATNO" in columns[name] else "NULL"
    visits = f"count(DISTINCT PATNO || '|' || {visit})" if visit and "PATNO" in columns[name] else "NULL"
    return (f"""SELECT {position} AS position, '{name}' AS "table", {flags}, count(*) AS "rows", {participants} AS participants, """
            f"""{visits} AS "participant-visits", {len(columns[name])} AS "columns" FROM "{name}\"""")


p.db.execute("CREATE OR REPLACE TEMP TABLE profile AS " + " UNION ALL ".join(grain(i, name) for i, name in enumerate(tables)))
p.sql("""SELECT * EXCLUDE (position), round("rows" / "participant-visits", 2) AS "rows per visit" FROM profile ORDER BY position""")

## 2. How the tables connect

Each table is linked to the keys it can be joined on, and two tables linked to the same key join on it. A table with a visit key also joins on `PATNO` alone, so only its visit link is drawn. Image tables carry both an image ID and a visit. Numbers in brackets are participants.

The table below the figure checks every link against a registry: `participant_status` for participants, `age_at_visit` for visits (it records age at every visit) and the image IDs in `images` and `image_files` for images. `share` is the fraction of a table's keys that the registry knows: a share well below 100% means rows drop out when you join.

In [ ]:
p.db.execute(r"""
    CREATE OR REPLACE TEMP VIEW image_index AS
    SELECT regexp_replace(trim(image_id), '^I|\.0+$', '', 'g') AS IMAGEID FROM images WHERE image_id IS NOT NULL
    UNION SELECT regexp_replace(trim(image_id), '^I|\.0+$', '', 'g') FROM image_files WHERE image_id IS NOT NULL
""")
IMAGE = r"regexp_replace(trim({}), '^I|\.0+$', '', 'g')"
REGISTRIES = {
    "participant": ("PATNO", "participant_status", ["PATNO"]),
    "visit": ("PATNO + EVENT_ID", "age_at_visit", ["PATNO", "EVENT_ID"]),
    "image": ("IMAGEID", "image_index", ["IMAGEID"]),
}
LAYERS = ["participant and visit tables", "keys", "image tables", "reference"]


def named(name, key):
    return {"EVENT_ID": "event", "IMAGEID": "image_id"}.get(key, key) if name in ("images", "image_files") else key


kinds = {name: [kind for kind, (key, registry, keys) in REGISTRIES.items() if all(named(name, k) in columns[name] for k in keys)]
         for name in tables}
p.db.execute("CREATE OR REPLACE TEMP TABLE edges (source VARCHAR, target VARCHAR, kind VARCHAR, key VARCHAR)")
p.db.executemany("INSERT INTO edges VALUES (?, ?, ?, ?)", [
    (name, REGISTRIES[kind][0], kind, REGISTRIES[kind][0]) for name in tables for kind in kinds[name]
    if not (kind == "participant" and "visit" in kinds[name])])
p.db.execute("CREATE OR REPLACE TEMP TABLE nodes (name VARCHAR, layer VARCHAR, position INTEGER)")
p.db.executemany("INSERT INTO nodes VALUES (?, ?, ?)", [
    *[(name, LAYERS[2] if "image" in kinds[name] else LAYERS[0] if kinds[name] else LAYERS[3], i) for i, name in enumerate(tables)],
    *[(key, LAYERS[1], len(tables) + i) for i, (key, registry, keys) in enumerate(REGISTRIES.values())]])
plot.relations(
    p.sql("""SELECT n.name, n.layer, f.participants FROM nodes n LEFT JOIN profile f ON f."table" = n.name
             ORDER BY list_position(?, n.layer), n.position""", LAYERS),
    p.sql("SELECT * FROM edges ORDER BY rowid"), "How the downloaded tables connect")

In [ ]:
def link(name, kind):
    """How many of a table's distinct keys are found in the key's registry."""
    key, registry, keys = REGISTRIES[kind]
    mine = [IMAGE.format(named(name, k)) if k == "IMAGEID" else named(name, k) for k in keys]
    return f"""
        SELECT '{name}' AS "table", '{key}' AS key, '{registry}' AS registry, count(*) AS keys, count(r.k0) AS matched
        FROM (SELECT DISTINCT {", ".join(f"{e} AS k{i}" for i, e in enumerate(mine))} FROM "{name}") t
        LEFT JOIN (SELECT DISTINCT {", ".join(f"{k} AS k{i}" for i, k in enumerate(keys))} FROM {registry}) r
               ON {" AND ".join(f"t.k{i} = r.k{i}" for i in range(len(keys)))}
        WHERE {" AND ".join(f"t.k{i} IS NOT NULL" for i in range(len(keys)))}"""


links = [link(name, kind) for name in tables for kind in kinds[name]
         if name != REGISTRIES[kind][1] and REGISTRIES[kind][1] in set(p.files) | {"image_index"}]
p.sql(f"""SELECT *, printf('%.1f%%', 100.0 * matched / keys) AS share FROM ({" UNION ALL ".join(links)}) ORDER BY key, "table" """)

## 3. Shared participants

How many participants each pair of tables has in common; the diagonal is each table's own count. Measures from two tables can only be analysed together for the participants they share.

In [ ]:
people = [name for name in tables if "PATNO" in columns[name]]
p.db.execute("CREATE OR REPLACE TEMP TABLE membership AS " + " UNION ALL ".join(
    f"""SELECT DISTINCT '{name}' AS tbl, PATNO FROM "{name}" WHERE PATNO IS NOT NULL""" for name in people))
listed = ", ".join(f"'{name}'" for name in people)
plot.heatmap(p.sql(f"""
    WITH pairs AS (SELECT a.tbl AS a, b.tbl AS b, count(*) AS n FROM membership a JOIN membership b USING (PATNO) GROUP BY ALL),
    grid AS (SELECT t.tbl, pairs.b, pairs.n FROM (SELECT unnest([{listed}]) AS tbl) t LEFT JOIN pairs ON pairs.a = t.tbl)
    SELECT * FROM (PIVOT grid ON b IN ({listed}) USING first(n) GROUP BY tbl) ORDER BY list_position([{listed}], tbl)
"""), "Participants shared by each pair of tables")

## 4. Visits each table covers

Participants at each visit code, for every table with a visit. In the original schedule V04, V06, V08, V10 and V12 fall at months 12, 24, 36, 48 and 60; later, remote and unscheduled visits have other codes. Rare codes are left out of the figure and listed below it.

In [ ]:
visit_tables = [name for name in sorted(p.files) if {"PATNO", "EVENT_ID"} <= columns[name]]
p.db.execute("CREATE OR REPLACE TEMP TABLE by_visit AS " + " UNION ALL ".join(
    f"""SELECT '{name}' AS tbl, EVENT_ID, count(DISTINCT PATNO) AS n FROM "{name}" WHERE EVENT_ID IS NOT NULL GROUP BY ALL"""
    for name in visit_tables))
ORDER = "CASE EVENT_ID WHEN 'SC' THEN 0 WHEN 'BL' THEN 1 ELSE 2 END, NOT starts_with(EVENT_ID, 'V'), length(EVENT_ID), EVENT_ID"
COMMON = "max(n) >= 0.02 * (SELECT max(n) FROM by_visit)"
common = p.column(f"SELECT EVENT_ID FROM by_visit GROUP BY EVENT_ID HAVING {COMMON} ORDER BY {ORDER}")
print("Left out:", ", ".join(p.column(f"SELECT EVENT_ID FROM by_visit GROUP BY EVENT_ID HAVING NOT ({COMMON}) ORDER BY {ORDER}")) or "nothing")
plot.heatmap(p.sql(f"""
    PIVOT by_visit ON EVENT_ID IN ({", ".join("'" + code.replace("'", "''") + "'" for code in common)}) USING first(n)
    GROUP BY tbl ORDER BY tbl
"""), "Participants at each visit, by table")

## 5. How the clean layer is built

The pipeline in `code/ppmi_pipeline` reads the raw tables and builds typed tables, then views on top of them. This figure is read from the pipeline's SQL (`schema.sql` and `views.sql`), so it always matches the code. Arrows point from a source to what is built from it, and numbers in brackets are participants.

In [ ]:
present = {sources.source_for(name): name for name in p.files if sources.source_for(name)}


def node(name):
    """A lineage name as a table: a raw input by the file it was read from, or a clean table."""
    if name.startswith("raw:"):
        return present.get(name[4:], f"{sources.SOURCES[name[4:]].prefixes[0]} (not downloaded)")
    return f"clean.{name}"


p.db.execute("CREATE OR REPLACE TEMP TABLE built (source VARCHAR, target VARCHAR, step INTEGER)")
p.db.executemany("INSERT INTO built VALUES (?, ?, ?)", [(node(s), f"clean.{t}", i) for i, (s, t) in enumerate(pipeline.lineage())])
with_patients = p.column("SELECT DISTINCT 'clean.' || table_name FROM information_schema.columns "
                         "WHERE table_catalog = 'clean' AND column_name = 'patient_id' ORDER BY 1")
counts = " UNION ALL ".join(f"SELECT '{name}' AS name, count(DISTINCT patient_id) AS participants FROM {name}" for name in with_patients)
plot.relations(p.sql(f"""
    WITH named AS (
        SELECT name, min(position) AS position FROM (
            SELECT source AS name, step AS position FROM built WHERE source NOT IN (SELECT target FROM built)
            UNION ALL SELECT target, 100000 + step FROM built
        ) GROUP BY name
    ), counts AS ({counts or "SELECT NULL::VARCHAR AS name, NULL::BIGINT AS participants WHERE false"}),
    types AS (SELECT 'clean.' || table_name AS name, table_type FROM information_schema.tables WHERE table_catalog = 'clean')
    SELECT n.name,
           CASE WHEN t.table_type = 'VIEW' THEN 'clean views' WHEN starts_with(n.name, 'clean.') THEN 'clean tables' ELSE 'raw tables' END AS layer,
           coalesce(c.participants, f.participants, 0) AS participants
    FROM named n LEFT JOIN types t USING (name) LEFT JOIN counts c USING (name) LEFT JOIN profile f ON f."table" = n.name
    ORDER BY CASE layer WHEN 'raw tables' THEN 0 WHEN 'clean tables' THEN 1 ELSE 2 END, n.position
"""), p.sql("SELECT source, target, 'built from' AS kind, 'pipeline SQL' AS key FROM built ORDER BY step"),
    "How the clean layer is built from the raw tables")

In [ ]:
p.sql("""
    SELECT 'clean.' || t.table_name AS "table", t.table_type AS kind, string_agg(b.source, ', ' ORDER BY b.step) AS "built from"
    FROM information_schema.tables t LEFT JOIN built b ON b.target = 'clean.' || t.table_name
    WHERE t.table_catalog = 'clean' GROUP BY ALL ORDER BY kind, "table"
""")

## 6. Every column

The columns of every table in both layers. For what each raw column means, `dictionary_columns` holds PPMI's description of every column once you have the Data Dictionary (Annotated); `dataset.ipynb` explains it.

In [ ]:
p.sql("""
    SELECT CASE table_catalog WHEN 'memory' THEN 'main' ELSE table_catalog END AS layer, table_name AS "table",
           string_agg(column_name, ', ' ORDER BY ordinal_position) AS columns
    FROM information_schema.columns WHERE table_catalog IN ('memory', 'clean') AND table_schema = 'main'
    GROUP BY ALL ORDER BY layer, "table"
""")

To read one table's columns with their descriptions, query `dictionary_columns`, for example `p.sql("""SELECT * FROM dictionary_columns WHERE "table" = 'mds_updrs_part_iii'""")`.